# Ke Jin — alert policy v1 and automated checks

**Owner:** Ke Jin. **Model:** LightGBM `no_dc` of run `aki_ml_v1_28e0b1fb83`, Platt calibration (fixed; do not retrain).
**Starting point:** the locked Demo I policy `demo-v1` in `artifacts/demo_i/policy.json` — alert when the 24h risk
is >= 0.21, every hour of ICU hours 1-72, 6 h cooldown. On validation it detects 76% of AKI events a median 12.7 h
early, but only about 1 in 3 alerts is followed by AKI and 37% of patients who never develop AKI get an alert.

This notebook has two tasks (README "Notebook ownership", notebook 03 Parts 2, 3 and 8):

| Part | Task | Output |
|---|---|---|
| 0 | Reproduce `demo-v1` from the shipped predictions (must pass first) | assertion only |
| A | Automated artifact and leakage checks | `src/aki_eval/checks.py` + pytest, check table |
| B | Alert policy v1: fewer false alerts at similar detection | `policy_v1_candidate.json`, grid and confirmation tables |

**Data roles (same as Demo I)**
- *Selection*: out-of-fold predictions of the training patients (`prediction_set == "train_oof"`). Every policy
  choice is made here.
- *Confirmation*: validation patients, used **once**, on the locked choice, at the end of Part B.
- *Test*: not shipped and never used here.

**Ground rules**
1. Write the candidate grid and the selection rule into `PRESPEC` (Part B.1) and commit it **before** running the
   grid. Do not change it after seeing validation results; a change starts a new, labelled version.
2. Do not edit `src/aki_ml/` (its hash is recorded by notebook 02; the dashboard loads the pickled models from it).
   Put new code in `src/aki_eval/` (create the package).
3. Outputs go to `artifacts/handoff/kejin/` and contain aggregate numbers only - never patient-level rows, IDs or
   timestamps. Clear cell outputs that show patient-level data before committing.
4. Hand back through a GitHub pull request: this notebook (outputs cleared), `src/aki_eval/`, `tests/`, and the
   aggregate CSV/JSON/PNG files.

In [ ]:
import hashlib
import json
import sys
import time
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs" / "ml.yaml").exists() and (path / "pipeline").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the AIForHealth-AKIEarlyWarning repository.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
from aki_ml import alerts as al

RUN_ID = "aki_ml_v1_28e0b1fb83"
FAMILY, FEATURE_SET, RISK = "lightgbm", "no_dc", "risk_platt"
HORIZONS = [6, 12, 24, 48]
DATA_ROOT = (REPO_ROOT / ".." / "Cleaned").resolve()
PRED_PATH = REPO_ROOT / "artifacts" / "predictions" / f"{RUN_ID}.parquet"
POLICY_PATH = REPO_ROOT / "artifacts" / "demo_i" / "policy.json"
SPLIT_PATH = DATA_ROOT / "splits" / "patient_split.parquet"
# The hand-off copy ships validation rows only; the full local dataset is filtered to validation on read.
SNAP_PATH = next(p for p in [DATA_ROOT / "datasets" / "snapshot_dataset_validation.parquet",
                             DATA_ROOT / "datasets" / "snapshot_dataset.parquet"] if p.exists())
OUT = REPO_ROOT / "artifacts" / "handoff" / "kejin"
OUT.mkdir(parents=True, exist_ok=True)

# The pickled models and the reproduced numbers depend on these exact versions (training environment).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "pyarrow": "19.0.0", "scikit-learn": "1.6.1", "lightgbm": "4.7.0"}
wrong = {k: metadata.version(k) for k, v in PINNED.items() if metadata.version(k) != v}
assert not wrong, f"package versions differ from the training environment: {wrong}"
DEMO_V1 = json.loads(POLICY_PATH.read_text(encoding="utf-8"))
print("Repository:", REPO_ROOT)
print("Data:", SNAP_PATH.relative_to(DATA_ROOT.parent))
print("Starting policy:", DEMO_V1["policy_version"], "-", DEMO_V1["threshold"]["selection_basis"][:110], "...")

## Part 0 — Reproduce `demo-v1` (must pass before any new work)

Recomputes the locked policy's train-OOF and validation metrics from the shipped predictions with
`src/aki_ml/alerts.py`. If this fails, the environment or the data copy is wrong; stop and report.

In [ ]:
def load_predictions(prediction_set, horizons=HORIZONS):
    t = pq.read_table(PRED_PATH, filters=[("family", "=", FAMILY), ("feature_set", "=", FEATURE_SET),
                                          ("prediction_set", "=", prediction_set), ("horizon_h", "in", horizons)],
                      columns=["split", "horizon_h", "patient_id", "hours_since_icu", "snapshot_time",
                               "creatinine_aki_onset_time", "eligible", "label", "risk_platt", "risk_isotonic"])
    d = t.to_pandas()
    d["patient_id"] = d["patient_id"].astype(str)
    return d


sel = load_predictions("train_oof")      # selection data, all four horizons (multi-horizon rules need them)
val = load_predictions("validation")     # confirmation data: use only in Part 0 and Part B.3
assert set(sel["split"]) == {"train"} and set(val["split"]) == {"validation"}, "test rows must never be loaded"
assert not set(sel["patient_id"]) & set(val["patient_id"])

P = DEMO_V1["alert_policy"]
BASE_POLICY = al.Policy(P["sustain_hours"], P["cooldown_hours"], P["rise_ratio"], P["rise_window_hours"])
BASE_THRESHOLD, H = DEMO_V1["threshold"]["value"], DEMO_V1["alert_horizon_h"]
WINDOW = tuple(DEMO_V1["monitoring_hours"])


def alert_frame(d, horizon=H, rise_window=12):
    d = d[(d["horizon_h"] == horizon) & d["hours_since_icu"].between(*WINDOW)].rename(columns={RISK: "risk"})
    d = al.prepare(d, horizon)
    d["prior_min"] = al.prior_min(d, rise_window)
    return d


sel_a, val_a = alert_frame(sel), alert_frame(val)
for name, frame, expected in [("train OOF", sel_a, DEMO_V1["selection_metrics_train_oof"]),
                              ("validation", val_a, DEMO_V1["validation_metrics"])]:
    got = al.evaluate(frame, BASE_THRESHOLD, BASE_POLICY, DEMO_V1["threshold"]["comparison"])
    bad = {k: (got[k], v) for k, v in expected.items() if not np.isclose(got[k], v, rtol=1e-9, atol=1e-12)}
    assert not bad, f"{name}: demo-v1 not reproduced: {bad}"
    print(f"{name}: demo-v1 reproduced - detection {got['event_detection_rate']:.3f}, PPV {got['alert_ppv']:.3f}, "
          f"{got['alerts_per_patient_day']:.2f} alerts/patient-day, non-AKI patients alerted {got['non_aki_patients_alerted']:.2f}")

## Part A — Automated artifact and leakage checks (notebook 03 Parts 2-3)

Goal: one reusable function per check, so notebook 03 can run the same code on the test split later without
edits. Put them in `src/aki_eval/checks.py`, each taking data frames/paths as arguments (never a hard-coded split)
and returning `(passed: bool, detail: str)`. Add `tests/test_checks.py` (pytest) with small **synthetic** frames:
one case that passes and one deliberately broken case that must fail for every check.

| # | Check | Inputs in this hand-off | Pass when |
|---|---|---|---|
| A1 | Unique keys | predictions: (family, feature_set, horizon_h, prediction_set, patient_id, hours_since_icu); snapshots: (stay_id, snapshot_time) | no duplicates |
| A2 | Probability bounds | `risk_raw`, `risk_platt`, `risk_isotonic` | finite and in [0, 1] |
| A3 | Schema and versions | `dataset_manifest.json` (columns, dataset version), `run_manifest.json`, `policy.json` model hashes | every listed column present with the recorded dtype; model file sha256 equals `policy.json` |
| A4 | Missing predictions | snapshots x horizons vs predictions | every eligible validation snapshot has a prediction for each horizon (count and list the gaps by horizon) |
| A5 | Split disjointness | `patient_split.parquet` | each `patient_id` in exactly one split; snapshot and prediction `split` equal the manifest split |
| A6 | One stay per patient | `patient_split.parquet`, snapshots | `patient_id` <-> `icustay_id` one-to-one |
| A7 | Chronology | snapshots | `icu_intime <= snapshot_time < censoring_time`; `hours_since_icu` consistent with the two times |
| A8 | Label definition | snapshots, predictions | for each horizon h: label 1 iff onset in (t, t + h]; eligible iff label 1 or follow-up >= h; prediction labels equal snapshot labels |
| A9 | No snapshot at or after AKI onset | snapshots | `snapshot_time < creatinine_aki_onset_time` whenever an onset exists |
| A10 | Features use only the past | `event_stream_validation.parquet` + `src/aki_ml/features.py` | recomputing features at t from events with time <= t reproduces the stored features (sample >= 2,000 rows; same tolerance as notebook 02: rtol 1e-5, atol 1e-4); then add one event just after t and confirm no feature at t changes |
| A11 | No post-snapshot urine in predictors | as A10 | urine features at t unchanged when all urine events after t are removed |
| A12 | No test information in fitting | `run_manifest.json` (`test_split`, preprocessing, calibration sections) | manifest records that the test split was never loaded and calibration was fit on train OOF |

Report every check, including the ones that pass, as one table saved to `artifacts/handoff/kejin/checks.csv`.
`require()` below stops the notebook when any check fails.

In [ ]:
snap = pq.read_table(SNAP_PATH, filters=[("split", "=", "validation")]).to_pandas()
split_manifest = pd.read_parquet(SPLIT_PATH)
CHECKS = []


def record(check_id, name, passed, detail=""):
    CHECKS.append({"id": check_id, "check": name, "passed": bool(passed), "detail": detail})


def require():
    table = pd.DataFrame(CHECKS)
    table.to_csv(OUT / "checks.csv", index=False)
    print(table.to_string(index=False))
    failed = table[~table["passed"]]
    assert failed.empty, f"{len(failed)} check(s) failed: {list(failed['id'])}"


# Two worked examples showing the expected shape. Move them into src/aki_eval/checks.py with the others.
risk_cols = ["risk_platt", "risk_isotonic"]
r = val[risk_cols].to_numpy(np.float64)
record("A2", "probability bounds (validation, no_dc)", np.isfinite(r).all() and ((r >= 0) & (r <= 1)).all(),
       f"{r.size:,} values")
per_patient_splits = split_manifest.groupby("patient_id")["split"].nunique()
record("A5", "each patient_id in exactly one split", (per_patient_splits == 1).all() and split_manifest["patient_id"].is_unique,
       f"{len(split_manifest):,} patients")

# TODO (Ke Jin): implement A1-A12 in src/aki_eval/checks.py, import them here, and call record() for each.
require()

## Part B — Alert policy v1 (notebook 03 Part 8)

**Goal.** Cut false alerts and the share of non-AKI patients who are alerted, while keeping AKI event
detection and lead time close to `demo-v1`. The model and its risk scores stay fixed; only the rule that turns
hourly risk into alerts changes.

**Candidate components** (choose a limited set; each needs an exact definition of when an alert starts,
repeats and ends):
- *Sustained high risk*: risk >= threshold for k consecutive hours (already in `alerts.Policy`).
- *Rapid risk increase*: risk >= ratio x the minimum of the prior W hours (already in `alerts.Policy`).
- *Multi-horizon escalation*: e.g. alert when 24h risk >= T24 **and** 12h or 48h risk agrees; or a two-level
  alert (watch / alert) from different horizons.
- *Stricter early threshold*: a higher threshold in the first ICU hours, where data are sparse.
- *Cooldown and repeat suppression*: cooldown length; no repeat while risk stays above threshold unless it
  rises by a set amount.

**Metrics** (`alerts.summarize`, unchanged definitions): event detection rate, alert PPV, alerts and false alerts
per patient-day, median lead time, share of non-AKI patients alerted. A new component goes in
`src/aki_eval/` and must reproduce `demo-v1` exactly when switched off.

### B.1 Prespecification (fill in, agree with the project owner, commit, then run B.2)

In [ ]:
PRESPEC = {
    "version": "v1-candidate",
    "selection_data": "train_oof",
    "alert_horizon_h": 24,
    "monitoring_hours": list(WINDOW),
    "thresholds": [round(x, 2) for x in np.arange(0.10, 0.41, 0.01)],
    # TODO (Ke Jin): list the policy candidates (component settings) to compare. Keep the grid small.
    "candidates": [],
    # Proposed rule; change it only before running B.2 and only with the project owner's agreement.
    "selection_rule": {
        "objective": "minimise false alerts per patient-day",
        "constraints": {"min_event_detection_rate": 0.70, "min_median_lead_h": 6.0, "min_ppv": 1 / 3,
                        "max_alerts_per_patient_day": 1.0},
        "tie_break": "fewer non-AKI patients alerted, then the simpler policy, then the higher threshold",
    },
    "baseline": {"policy": DEMO_V1["policy_version"], "threshold": BASE_THRESHOLD, "alert_policy": DEMO_V1["alert_policy"]},
}
(OUT / "prespec.json").write_text(json.dumps(PRESPEC, indent=2), encoding="utf-8")
print(json.dumps(PRESPEC["selection_rule"], indent=2))

### B.2 Selection on train OOF

Run every candidate x threshold on `sel` (train OOF). Save the full grid to `policy_grid_train_oof.csv` and a
one-row-per-candidate summary (best operating point under the constraints) to `policy_candidates_train_oof.csv`.
Show the detection vs alert-burden trade-off for `demo-v1` and the best candidates in one figure.

In [ ]:
# TODO (Ke Jin): build the grid from PRESPEC["candidates"], evaluate on sel only, apply PRESPEC["selection_rule"].
# For multi-horizon rules, sel already holds all four horizons (join them on patient_id + hours_since_icu).
CHOSEN = None   # the selected candidate (dict of component settings + threshold)

### B.3 Confirmation on validation (run once, after B.2 is locked)

Evaluate `CHOSEN` and `demo-v1` on `val` with 1,000 patient-level bootstrap resamples (`alerts.bootstrap`, seed
20261102). Save `policy_confirmation_validation.csv`, then write `policy_v1_candidate.json` with the same fields as
`policy.json` (`policy_version: "v1-candidate"`, `threshold.locked: false`; the owner locks it). Do not tune
anything after this cell has run.

In [ ]:
# TODO (Ke Jin): confirmation table (CHOSEN vs demo-v1, point estimates and 95% CIs) and policy_v1_candidate.json.

## Hand-back checklist

- [ ] Part 0 passes from a clean kernel.
- [ ] `src/aki_eval/checks.py` implements A1-A12; `pytest tests/test_checks.py` passes; `checks.csv` has no failure.
- [ ] `prespec.json` was committed before the B.2 results (commit order visible in the PR).
- [ ] `policy_grid_train_oof.csv`, `policy_candidates_train_oof.csv`, `policy_confirmation_validation.csv`,
      `policy_v1_candidate.json` and the trade-off figure are in `artifacts/handoff/kejin/`.
- [ ] A short written summary (5-10 lines) at the end of this notebook: chosen rule, what it costs and gains
      versus `demo-v1` on validation, and known limitations.
- [ ] Notebook outputs cleared of anything patient-level.